In [3]:
import pandas as pd

raw_df = pd.read_csv(
    "./data/payment_transactions_raw.csv",
    parse_dates=["transaction_date"]    
)
raw_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   transaction_id     30000 non-null  object        
 1   customer_id        30000 non-null  object        
 2   merchant_id        30000 non-null  object        
 3   transaction_date   30000 non-null  datetime64[ns]
 4   payment_method     30000 non-null  object        
 5   amount             30000 non-null  float64       
 6   channel            30000 non-null  object        
 7   merchant_category  30000 non-null  object        
 8   status             30000 non-null  object        
 9   failure_reason     3663 non-null   object        
dtypes: datetime64[ns](1), float64(1), object(8)
memory usage: 2.3+ MB


In [12]:
raw_df.head()

,transaction_id,customer_id,merchant_id,transaction_date,payment_method,amount,channel,merchant_category,status,failure_reason
0,TXN0002574,CUST03596,MER0660,2026-01-01 00:02:00,Credit Card,154.94,POS,Entertainment,Success,NaN
1,TXN0019468,CUST07590,MER0662,2026-01-01 00:02:00,Bank Transfer,404.01,Mobile App,Entertainment,Success,NaN
2,TXN0012951,CUST04006,MER0769,2026-01-01 00:05:00,Credit Card,377.06,Web,Food & Beverage,Success,NaN
3,TXN0026531,CUST06896,MER0125,2026-01-01 00:05:00,Debit Card,338.75,Mobile App,Retail,Success,NaN
4,TXN0000458,CUST00816,MER0256,2026-01-01 00:09:00,Bank Transfer,607.64,POS,Digital Services,Success,NaN


### **duplicated rows check**

In [11]:
raw_df.duplicated().sum()

np.int64(0)

In [14]:
raw_df["transaction_id"].duplicated().sum()

np.int64(0)

### **missing values check**

In [10]:
raw_df.isna().sum()

transaction_id           0
customer_id              0
merchant_id              0
transaction_date         0
payment_method           0
amount                   0
channel                  0
merchant_category        0
status                   0
failure_reason       26337
dtype: int64

### **status destribution**

In [13]:
trnx_distribution_by_status = (
    raw_df
    .groupby("status")
    .agg(
        trnx_count = ("transaction_id", "nunique")
    )
)

trnx_distribution_by_status

,trnx_count
status,
Failed,3663
Success,26337


In [15]:
raw_df.groupby("status")["failure_reason"].apply(
    lambda s: s.isna().sum()
)

### **transaction_date values check**

In [17]:
transaction_date_dq_check = {
    "Min": raw_df["transaction_date"].min(),
    "Max": raw_df["transaction_date"].max()
}
transaction_date_dq_check

{'Min': Timestamp('2026-01-01 00:02:00'),
 'Max': Timestamp('2026-03-30 23:58:00')}

### **amount quality check**

In [18]:
amount_date_dq_check = {
    "Min": raw_df["amount"].min(),
    "Max": raw_df["amount"].max()
}
amount_date_dq_check

{'Min': np.float64(20.0), 'Max': np.float64(10000.0)}

### **values check**

In [19]:
for col in ["payment_method", "channel", "merchant_category", "status"]:
    print(f"\n{col}:")
    print(raw_df[col].value_counts(dropna=False))


payment_method:
payment_method
Credit Card      9524
Debit Card       8409
Mobile Wallet    7628
Bank Transfer    4439
Name: count, dtype: int64

channel:
channel
Mobile App    13453
Web           10554
POS            5993
Name: count, dtype: int64

merchant_category:
merchant_category
Retail              6672
Food & Beverage     5347
Travel              4512
Entertainment       3590
Utilities           3514
Digital Services    3315
Healthcare          3050
Name: count, dtype: int64

status:
status
Success    26337
Failed      3663
Name: count, dtype: int64
